In [ ]:
%pip install                                                         \
        numpy                 matplotlib          matplotlib-inline  \
		ipython                                                      \
		sentencepiece         protobuf            polars             \
        sentence-transformers tiktoken            fastembed

%restart_python

In [ ]:
import os
import gc
import ast
import numpy as np
import scipy as sp
import torch
import torch.nn as nn
from torch.utils.data      import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer

import polars as pl
import pandas as pd
from pyspark.sql           import SparkSession
from pyspark.sql.functions import col
from delta                 import configure_spark_with_delta_pip

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
builder = SparkSession.builder\
            .config("spark.sql.sources.commitProtocolClass", "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")\
            .config("spark.sql.parquet.output.committer.class", "org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter")\
            .config("spark.mapreduce.fileoutputcommitter.marksuccessfuljobs","false")\
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")\
            .config("spark.sql.adaptive.enabled", True)\
            .config("spark.sql.shuffle.partitions", "auto")\
            .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "100MB")\
            .config("spark.sql.adaptive.coalescePartitions.enabled", True)\
            .config("spark.sql.dynamicPartitionPruning.enabled", True)\
            .config("spark.sql.autoBroadcastJoinThreshold", "10MB")\
            .config("spark.sql.session.timeZone", "Asia/Tokyo")\
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
            .config("spark.databricks.delta.write.isolationLevel", "SnapshotIsolation")\
            .config("spark.databricks.delta.optimizeWrite.enabled", True)\
            .config("spark.databricks.delta.autoCompact.enabled", True)
            # Delta Lake 用の SQL コミットプロトコルを指定
            # Parquet 出力時のコミッタークラスを指定
            # Azure Blob Storage (ABFS) 用のコミッターファクトリを指定
            # '_SUCCESS'で始まるファイルを書き込まないように設定
            # JavaSerializerからKryoSerializerへの変更
            # AQE(Adaptive Query Execution)の有効化
            # パーティション数を自動で調整するように設定
            # シャッフル後の1パーティションあたりの最小サイズを指定
            # AQEのパーティション合成の有効化
            # 動的パーティションプルーニングの有効化
            # 小さいテーブルのブロードキャスト結合への自動変換をするための閾値調整
            # SparkSessionのタイムゾーンを日本標準時刻に設定
            # Apache Arrow 形式で Pandas<=>Spark 変換を行う
            # Delta Lake固有のSQL構文や解析機能を拡張モジュールとして有効化
            # SparkカタログをDeltaLakeカタログへ変更
            # Delta Lake書き込み時のアイソレーションレベルを「スナップショット分離」に設定
            # 書き込み時にデータシャッフルを行い、大きなファイルを生成する機能の有効化
            # 書き込み後に小さなファイルを自動で統合する機能の有効化

# 追加の設定処理
# クラスターサイズによって、動的に書き換えること
builder = builder\
			.config("spark.driver.memory", "64g")\
    		.config("spark.driver.maxResultSize", "32g")\
			.config("spark.rpc.message.maxSize", "1024")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# メモリ管理
del builder
gc.collect()


In [ ]:
# ウェジットからの設定の読み込み
BRAND_NAME                = dbutils.widgets.get('BRAND_NAME')
TARGET_BRAND_WORDS        = dbutils.widgets.get('TARGET_BRAND_WORDS')
EXTRACTION_ID             = dbutils.widgets.get('EXTRACTION_ID')

# メモ：
# 環境変数・ウィジット経由でのパラメータの取得方法では、無条件にパラメータの型がStringに変換されてしまう
# そのため、受け取ったパラメータを適切に型変換する必要がある
EXTRACTION_ID             = int(EXTRACTION_ID)

# メモ：
# TARGET_BRAND_WORDS の中身として
# "['aaa', 'bbb', 'ccc']"
# "aaa" or "'aaaaa'"
# 上記のような「文字列」を想定している
# 
# このような文字列を正しくパースするために、astライブラリを利用することとした
try:
	TARGET_BRAND_WORDS = ast.literal_eval(TARGET_BRAND_WORDS)
	if isinstance(TARGET_BRAND_WORDS, str):
		TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]
except (ValueError, SyntaxError):
	TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]

# 簡易デバッグ用
print(f'BRAND_NAME:                {BRAND_NAME}')
print(f'TARGET_BRAND_WORDS:        {TARGET_BRAND_WORDS}')
print(f'EXTRACTION_ID:             {EXTRACTION_ID}')

In [ ]:
# 特定の地点に対するADIDリストを取得
POLYGONLIST_TABLE = "adinte_adintedmp.envprod.polygonlist"
sdf_polygonlist   = spark.read.table(POLYGONLIST_TABLE)\
							.select(['project', 'caption', 'extraction_id', 'branch_id', 'spot_id'])\
                            .filter(col('extraction_id') == EXTRACTION_ID)
ADIDLIST_TABLE    = "adinte_adintedmp.envprod.source"
sdf_adidlist      = spark.read.table(ADIDLIST_TABLE)\
							.select(['extraction_id', 'branch_id', 'spot_id', 'adid'])\
                            .filter(col('extraction_id') == EXTRACTION_ID)\
							.join(sdf_polygonlist, on=['extraction_id', 'branch_id', 'spot_id'], how='inner')\
                            .select(['caption', 'adid'])\
                            .dropDuplicates()

target_adidlist   = [row['adid']    for row in sdf_adidlist.select('adid').dropDuplicates().toLocalIterator()]
pdf_adidlist      = sdf_adidlist.toPandas()

# メモリ管理
sdf_polygonlist.unpersist()
sdf_adidlist.unpersist()
del sdf_polygonlist, sdf_adidlist
spark.catalog.clearCache()
gc.collect()

pdf_adidlist

In [ ]:
# メモ：
# cohort.npzは以下のようなデータ構成になっている
# cohort.npz
#     |-- data              : 計算済みコホート係数行列
#     |-- indices           : データの位置指定子(列)
#     |-- indptr            : 行ごとのスライスインデックス
#     |-- shape             : コホート係数行列の形(ADIDのリスト × コホートキャプションのリスト)
#     |-- adid_list         : ADIDのリスト(列)
#     |-- business_codelist : コホートキャプションIDのリスト(行)
TARGET      = "/Volumes/stgadintedmpadintedi/featurestore/behaviorvector/cohort.npz"
npz         = np.load(TARGET, allow_pickle=True)
np_cohort   = sp.sparse.csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
np_adidlist = npz["adid_list"]
np_codelist = npz["business_codelist"]

# 特定のADIDのみの抜き出し
indexer     = pd.Index(np_adidlist)
indices     = indexer.get_indexer(target_adidlist)
indices     = indices[indices != -1]
np_cohort   = np_cohort[indices, :]
np_adidlist = np_adidlist[indices]


# メモ：
# L2正規化を行うにあたって、疎行列専用に書く必要がある
# 密行列であれば簡単に書くことが可能であるが、今回の要件に適合しない
# そのため疎行列な対角行列を経由することにより、これを実現することとした
# 
# 当初すでにL2正規化は適用済みであったが、時々適用されていない状態になることがあるようだ
# そのため、改めて適用することとした
# L2正規化
if not np.allclose(np_cohort.power(2).sum(axis=1), 1.0):
	l2norm_vector = sp.sparse.linalg.norm(np_cohort, axis=1)
	np_cohort     = sp.sparse.diags(1 / np.maximum(l2norm_vector, 1e-10)) @ np_cohort

	# メモリ管理
	del l2norm_vector

# メモリ管理
del npz, indexer, indices
gc.collect()

print(f"コホート行列の形状 :        {np_cohort.shape}")
print(f"  - ADID数            : {np_cohort.shape[0]:,}")
print(f"  - コホートキャプション数 : {np_cohort.shape[1]:,}")
print(f"  - 非零要素数          : {np_cohort.nnz:,}")
print(f"  - 疎密度             : {np_cohort.nnz / (np_cohort.shape[0] * np_cohort.shape[1]):.6f}")

In [ ]:
# Embeddingモデルによる高次元空間への写像
# model            = SentenceTransformer('BAAI/bge-m3')
model            = SentenceTransformer('cl-nagoya/ruri-v3-30m')
model.half()
lp_matrix        = model.encode(TARGET_BRAND_WORDS, normalize_embeddings=True)  # キーワード数M × 1024

# メモリ管理
del model
gc.collect()

lp_matrix

In [ ]:
class TwoTowerModel(nn.Module):
    """ADID行動特徴と商品LP係数を共通の潜在空間に射影するTwo-Towerモデル。"""

    def __init__(self, action_dim:int, item_dim:int, embed_dim:int=128):
        super().__init__()

        # Action Tower: 行動ベクトル (ACTION_DIM → 1024 → 1024 → embed_dim)
        self.action_tower = nn.Sequential(
            nn.Linear(action_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, embed_dim),
        )

        # Item Tower: 商品ベクトル (ITEM_DIM → 1024 → 1024 → embed_dim)
        self.item_tower = nn.Sequential(
            nn.Linear(item_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, embed_dim),
        )

    def encode_user(self, action_features: torch.Tensor) -> torch.Tensor:
        """行動ベクトルをembed_dim次元に変換してL2正規化。"""
        a = self.action_tower(action_features)
        return nn.functional.normalize(a, p=2, dim=1)

    def encode_item(self, item_features: torch.Tensor) -> torch.Tensor:
        """LP係数ベクトルをembed_dim次元に変換してL2正規化。"""
        i = self.item_tower(item_features)
        return nn.functional.normalize(i, p=2, dim=1)

    def forward(self, action_features: torch.Tensor, item_features: torch.Tensor) -> torch.Tensor:
        a_embed = self.encode_user(action_features)
        i_embed = self.encode_item(item_features)
        # 内積 = L2正規化済みベクトルのコサイン類似度 (-1.0 ~ 1.0)
        inner   = torch.sum(a_embed * i_embed, dim=1)
        return torch.stack([inner, -inner], dim=1)



MODEL_PATH = "生成ファイル/two_tower_model.pt"
checkpoint = torch.load(MODEL_PATH, map_location='cpu')
loaded_model = TwoTowerModel(
    action_dim=checkpoint['action_dim'],
    item_dim=checkpoint['item_dim'],
    embed_dim=checkpoint['embed_dim']
)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval()

# パラメータ数の確認
total_params = sum(p.numel() for p in loaded_model.parameters())
print(f"モデルパラメータ数: {total_params:,}")
print(f"\nAction Tower:")
print(loaded_model.action_tower)
print(f"\nItem Tower:")
print(loaded_model.item_tower)

In [ ]:
all_preds = []
for idx in range(np_cohort.shape[0]):
	preds = model(np_cohort[idx], lp_matrix)
	all_preds.extend(preds.cpu().numpy())

all_preds       = np.array(all_preds)[:, 0]
indices         = np.argsort(all_preds)[::-1]
sorted_adidlist = np_adidlist[indices]
sorted_scored   = all_preds[indices]


# 閾値の設定(上位20%地点を指定する)
REASON_THRESHOLD = np.quantile(sorted_scored, 0.8)

# フレームワークへの変換
pldf_scores      = pl.LazyFrame({'adid': sorted_adidlist, 'score': sorted_scored})\
						.cast(  {'adid': pl.String,       'score': pl.Float32})\
                        .select(['adid', 'score'])
pldf_adidlist    = pl.from_pandas(pdf_adidlist).lazy()\
						.join(pldf_scores, on='adid', how='inner')\
                        .select(['caption', 'adid', 'score'])\
                        .filter(pl.col('score') > REASON_THRESHOLD)\
                        .collect()
# 解析対象地点毎のカウント数を計算
pldf_store_sum   = pldf_adidlist\
						.group_by(pl.col('caption'), maintain_order=True)\
                        .agg(pl.count().alias('total_count'))\
                        .select(['caption', 'total_count'])
count            = pldf_adidlist.select(pl.col("adid").n_unique()).item()

# メモリ管理
del pldf_scores
gc.collect()

print(f"処理対象        : {BRAND_NAME}")
print(f"ユニークなADID数 : {count}")
print(pldf_adidlist)



In [ ]:
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_all_with_two_tower.parquet"
pldf_adidlist.write_parquet( TARGET, compression="snappy")
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_separate_with_two_tower.parquet"
pldf_store_sum.write_parquet(TARGET, compression="snappy")